# TSE Analysis — Fase 1 PoC: Extração, Parsing e Materialização de Logs da Urna (`.jez` $\rightarrow$ `fact_urna_event`)

Este notebook valida de ponta a ponta o pipeline da **Fase 1 (`phase-1-urna-logs-poc`)** tanto localmente quanto em execução remota em **Kaggle Kernels**:
1. **Extração em Memória/Streaming de `.jez`**: Suporte a contêineres `7z/LZMA` (`py7zr`) e `ZIP`, ordenando segmentos rotacionados (`logd.dat.2` $\rightarrow$ `logd.dat.1` $\rightarrow$ `logd.dat`) sem deixar resíduos em disco.
2. **Parser Estrutural Linha-a-Linha**: Ordem canônica estritamente monotônica via `file_seq` (`Int64`), detecção de `has_timestamp_regression` (`Boolean`) sem reordenação de linhas e descarte imediato do autenticador hexadecimal (`mac_code`).
3. **Catálogo Semântico (`event_catalog`) e Sanitização Zero-PII**: Classificação por allowlist regex (habilitação, biometria com `biometric_attempt_num`, timeout de 45s, tecla indevida, computação de voto, suspensão/retomada/contingência e abertura/encerramento), convertendo qualquer mensagem fora da allowlist em `UNKNOWN_EVENT` com retenção zero de texto livre ou PII.
4. **Materialização Colunar em Parquet ZSTD e Consultas Analíticas**: Escrita particionada (`election_year=2026/turn=1/uf=SP/`) validada contra `FACT_URNA_EVENT_POLARS_SCHEMA` e `FACT_URNA_EVENT_ARROW_SCHEMA`, seguida de auditoria com **Polars** e **DuckDB**.

In [1]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = ["py7zr", "polars", "pyarrow", "duckdb", "pyyaml"]
missing = [pkg for pkg in REQUIRED_PACKAGES if importlib.util.find_spec(pkg.replace("pyyaml", "yaml")) is None]
if missing:
    print(f"Instalando pacotes ausentes no ambiente: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Todas as dependências (py7zr, polars, pyarrow, duckdb, pyyaml) já estão instaladas.")

Todas as dependências (py7zr, polars, pyarrow, duckdb, pyyaml) já estão instaladas.


In [2]:
from __future__ import annotations

import io
import json
import re
import tempfile
import zipfile
from datetime import datetime
from pathlib import Path

import duckdb
import polars as pl
import py7zr
import py7zr.io
import pyarrow as pa
import pyarrow.parquet as pq

# Caso executado localmente dentro do repositório tse-analysis, adiciona src/ ao sys.path
for candidate_root in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate_root / "src" / "tse_analysis").exists():
        sys.path.insert(0, str(candidate_root / "src"))
        break

try:
    from tse_analysis.config import ExecutionConfig
    from tse_analysis.logs import (
        FACT_URNA_EVENT_ARROW_SCHEMA,
        FACT_URNA_EVENT_POLARS_SCHEMA,
        SectionTurnContext,
        events_to_polars,
        events_to_pyarrow,
        extract_jez_events,
        write_fact_urna_event_partition,
    )
    EXECUTION_MODE = "local_package"
except ImportError:
    EXECUTION_MODE = "kaggle_standalone"

    FACT_URNA_EVENT_POLARS_SCHEMA = pl.Schema({
        "election_year": pl.Int16,
        "turn": pl.Int8,
        "uf": pl.String,
        "municipio_code": pl.Int32,
        "zone_num": pl.Int16,
        "section_num": pl.Int32,
        "urna_serial": pl.String,
        "file_seq": pl.Int64,
        "raw_timestamp": pl.Datetime("us"),
        "has_timestamp_regression": pl.Boolean,
        "application": pl.String,
        "log_level": pl.String,
        "event_code": pl.String,
        "biometric_attempt_num": pl.Int8,
    })

    FACT_URNA_EVENT_ARROW_SCHEMA = pa.schema([
        pa.field("election_year", pa.int16(), nullable=False),
        pa.field("turn", pa.int8(), nullable=False),
        pa.field("uf", pa.string(), nullable=False),
        pa.field("municipio_code", pa.int32(), nullable=False),
        pa.field("zone_num", pa.int16(), nullable=False),
        pa.field("section_num", pa.int32(), nullable=False),
        pa.field("urna_serial", pa.string(), nullable=False),
        pa.field("file_seq", pa.int64(), nullable=False),
        pa.field("raw_timestamp", pa.timestamp("us"), nullable=True),
        pa.field("has_timestamp_regression", pa.bool_(), nullable=False),
        pa.field("application", pa.string(), nullable=False),
        pa.field("log_level", pa.string(), nullable=False),
        pa.field("event_code", pa.string(), nullable=False),
        pa.field("biometric_attempt_num", pa.int8(), nullable=True),
    ])

    ALLOWLIST_RULES = [
        ("LOGD_STARTED", re.compile(r"(?i)(?:in[ií]cio\s+das\s+opera[cç][oõ]es\s+do\s+logd|logd\s+iniciado)"), False),
        ("SECTION_OPENED", re.compile(r"(?i)(?:urna\s+pronta\s+para\s+receber\s+votos|emiss[aã]o\s+d[ae]\s+zer[eé]sima|in[ií]cio\s+da\s+vota[cç][aã]o)"), False),
        ("VOTER_ENABLED", re.compile(r"(?i)^(?=.*\beleitor\s+(?:foi\s+)?habilitado\b)(?!.*\b(?:sem\s+biometria|manual)\b).*$"), False),
        ("BIOMETRIC_READ_ATTEMPT", re.compile(r"(?i)^(?!(?:.*\b(?:mes[aá]rio|presidente|autoteste|teste\s+de\s+sensor)\b))(?:.*(?:tentativa\s+de\s+(?:reconhecimento|leitura|identifica[cç][aã]o)\s+biom[eé]tric[oa]|solicita(?:ndo)?\s+digital[.:;]?\s*tentativa|capturada\s+a\s+digital[.:;]?\s*tentativa|n[uú]mero\s+de\s+tentativas\s+de\s+reconhecimento\s+do\s+dedo(?:[.:;]?\s*tentativa)?)\s*(?:n[uúº°.]*\s*)?[:=\-]?\s*[\[\(]?(?P<attempt>\d{1,3})[\]\)]?(?:\b|$))"), True),
        ("BIOMETRIC_NOT_RECOGNIZED", re.compile(r"(?i)^(?!(?:.*\b(?:mes[aá]rio|presidente|autoteste|teste\s+de\s+sensor)\b))(?:.*(?:(?:digital|biometria|dedo)\s+n[aã]o\s+reconhecid[oa]|timeout\s+de\s+reconhecimento\s+do\s+dedo))"), False),
        ("BIOMETRIC_RECOGNIZED", re.compile(r"(?i)^(?!(?:.*\b(?:mes[aá]rio|presidente|autoteste|teste\s+de\s+sensor)\b))(?:.*(?:(?<!n[aã]o\s)\b(?:biometria|digital|dedo)\s+reconhecid[oa]\b|tipo\s+de\s+habilita[cç][aã]o\s+do\s+eleitor\s*\[?biom[eé]trica\]?))"), False),
        ("NO_BIOMETRIC_REGISTERED", re.compile(r"(?i)^(?!(?:.*\b(?:mes[aá]rio|presidente)\b))(?:.*(?:n[aã]o\s+h[aá]\s+biometria|eleitor\s+sem\s+biometria|ano\s+de\s+nascimento\s+digitado\s+[eé]\s+igual\s+ao\s+do\s+cadastro))"), False),
        ("MANUAL_ENABLE_PATH", re.compile(r"(?i)(?:habilita[cç][aã]o\s+manual|libera[cç][aã]o\s+manual|eleitor\s+habilitado\s+sem\s+biometria|solicita[cç][aã]o\s+de\s+dado\s+pessoal\s+do\s+eleitor\s+para\s+habilita[cç][aã]o\s+manual|mes[aá]rio\s+\d+\s+habilitou\s+o\s+eleitor)"), False),
        ("INACTIVITY_TIMEOUT_45S", re.compile(r"(?i)(?:(?:eleitor\s+)?sem\s+atividade\s+por\s+45\s*s|inatividade\s+(?:de|por)\s+45\s*s|tempo\s+limite\s+de\s+inatividade)"), False),
        ("INVALID_KEY_PRESSED", re.compile(r"(?i)tecla\s+(?:indevida|inv[aá]lida)"), False),
        ("CARGO_VOTE_RECORDED", re.compile(r"(?i)voto\s+(?:confirmado|registrado)\s+para\b"), False),
        ("VOTER_VOTE_COMPUTED", re.compile(r"(?i)(?:o\s+)?voto\s+do\s+eleitor\s+(?:foi\s+)?computado"), False),
        ("VOTING_SUSPENDED", re.compile(r"(?i)vota[cç][aã]o\s+suspensa"), False),
        ("VOTING_RESUMED", re.compile(r"(?i)vota[cç][aã]o\s+retomada"), False),
        ("CONTINGENCY_EVENT", re.compile(r"(?i)(?:modo\s+de\s+conting[eê]ncia|substitui[cç][aã]o\s+de\s+urna)"), False),
        ("SECTION_CLOSED", re.compile(r"(?i)(?:encerramento\s+da\s+(?:vota[cç][aã]o|se[cç][aã]o)|gerando\s+arquivo\s+de\s+resultado\s*\[imgbu\.dat\]\s*\+\s*\[t[eé]rmino\])"), False),
    ]

print(f"Modo de execução ativo: {EXECUTION_MODE}")

Modo de execução ativo: local_package


In [3]:
# 1. Construção de pacotes .jez sintéticos (7z/LZMA com rotação logd.dat.2 -> logd.dat.1 -> logd.dat e ZIP)
PII_SENTINEL_CPF = "98765432100"
PII_SENTINEL_TITULO = "123456780199"
MAC_SENTINEL = "DEADBEEFCAFEBABE1234567890ABCDEF"

logd_rot_2 = (
    f"04/10/2026\t07:00:00\tINFO\t77889900\tscue\tInício das operações do logd\t{MAC_SENTINEL}\r\n"
    f"04/10/2026\t07:30:00\tINFO\t77889900\tVOTA\tEmissão da Zerésima\t{MAC_SENTINEL}\r\n"
).encode("latin1")

logd_rot_1 = (
    f"04/10/2026\t08:00:00\tINFO\t77889900\tVOTA\tUrna pronta para receber votos\t{MAC_SENTINEL}\r\n"
    f"04/10/2026\t08:01:05\tINFO\t77889900\tVOTA\tEleitor foi habilitado\t{MAC_SENTINEL}\r\n"
    f"04/10/2026\t08:01:10\tINFO\t77889900\tVOTA\tTentativa de reconhecimento biométrico 1\t{MAC_SENTINEL}\r\n"
    f"04/10/2026\t08:01:13\tWARN\t77889900\tVOTA\tDigital não reconhecida\t{MAC_SENTINEL}\r\n"
    f"04/10/2026\t08:01:20\tINFO\t77889900\tVOTA\tTentativa de reconhecimento biométrico 2\t{MAC_SENTINEL}\r\n"
    f"04/10/2026\t08:01:23\tINFO\t77889900\tVOTA\tBiometria reconhecida\t{MAC_SENTINEL}\r\n"
).encode("latin1")

logd_active = (
    f"04/10/2026\t08:02:08\tWARN\t77889900\tVOTA\tInatividade de 45 segundos\t{MAC_SENTINEL}\r\n"
    f"04/10/2026\t08:02:14\tWARN\t77889900\tVOTA\tTecla indevida pressionada\t{MAC_SENTINEL}\r\n"
    # Regressão de relógio proposital (08:01:55 < 08:02:14)
    f"04/10/2026\t08:01:55\tINFO\t77889900\tVOTA\tVoto confirmado para o cargo\t{MAC_SENTINEL}\r\n"
    f"04/10/2026\t08:02:30\tINFO\t77889900\tVOTA\tO voto do eleitor foi computado\t{MAC_SENTINEL}\r\n"
    f"04/10/2026\t08:05:00\tINFO\t77889900\tVOTA\tEleitor foi habilitado\t{MAC_SENTINEL}\r\n"
    f"04/10/2026\t08:05:04\tWARN\t77889900\tVOTA\tNão há biometria cadastrada para o eleitor\t{MAC_SENTINEL}\r\n"
    f"04/10/2026\t08:05:10\tINFO\t77889900\tVOTA\tHabilitação manual pelo mesário\t{MAC_SENTINEL}\r\n"
    f"04/10/2026\t08:05:35\tINFO\t77889900\tVOTA\tO voto do eleitor foi computado\t{MAC_SENTINEL}\r\n"
    # Linha fora da allowlist contendo PII simulado
    f"04/10/2026\t12:00:00\tINFO\t77889900\tVOTA\tOperação não catalogada CPF {PII_SENTINEL_CPF} Titulo {PII_SENTINEL_TITULO}\t{MAC_SENTINEL}\r\n"
    f"04/10/2026\t17:00:00\tINFO\t77889900\tVOTA\tEncerramento da votação\t{MAC_SENTINEL}\r\n"
).encode("latin1")

jez_7z_buf = io.BytesIO()
with py7zr.SevenZipFile(jez_7z_buf, mode="w") as archive:
    archive.writestr(logd_active, "logd.dat")
    archive.writestr(logd_rot_1, "logd.dat.1")
    archive.writestr(logd_rot_2, "logd.dat.2")
jez_7z_bytes = jez_7z_buf.getvalue()

jez_zip_buf = io.BytesIO()
with zipfile.ZipFile(jez_zip_buf, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("logd.dat", logd_active)
jez_zip_bytes = jez_zip_buf.getvalue()

print(f"Pacote .jez 7z/LZMA gerado em memória: {len(jez_7z_bytes)} bytes (magic={jez_7z_bytes[:6]!r})")
print(f"Pacote .jez ZIP gerado em memória: {len(jez_zip_bytes)} bytes (magic={jez_zip_bytes[:4]!r})")

Pacote .jez 7z/LZMA gerado em memória: 669 bytes (magic=b"7z\xbc\xaf'\x1c")
Pacote .jez ZIP gerado em memória: 449 bytes (magic=b'PK\x03\x04')


In [4]:
# 2. Extração em memória, parsing estrutural, classificação por allowlist e materialização Parquet ZSTD
work_dir = Path(tempfile.mkdtemp(prefix="tse_urna_logs_poc_"))

if EXECUTION_MODE == "local_package":
    ctx = SectionTurnContext(
        election_year=2026,
        turn=1,
        uf="SP",
        municipio_code=71072,
        zone_num=1,
        section_num=101,
    )
    extraction = extract_jez_events(jez_7z_bytes, section_context=ctx, availability_status="AVAILABLE")
    df_events = events_to_polars([extraction])
    arrow_events = events_to_pyarrow([extraction])
    cfg = ExecutionConfig(election_year=2026, turn=1, uf="SP", data_dir=work_dir)
    parquet_path = write_fact_urna_event_partition(df_events, config=cfg)
else:
    with py7zr.SevenZipFile(io.BytesIO(jez_7z_bytes), mode="r") as archive:
        if hasattr(py7zr.io, "BytesIOFactory"):
            factory = py7zr.io.BytesIOFactory(limit=64 * 1024 * 1024)
            archive.extract(factory=factory)
            members = {}
            for k, bio in factory.products.items():
                bio.seek(0)
                members[k] = bio.read()
        else:
            read_fn = getattr(archive, "read", None) or getattr(archive, "readall", None)
            members = {k: bio.read() for k, bio in read_fn().items()}

    def _rot_key(name: str) -> tuple[int, str]:
        m = re.search(r"\.(\d+)$", name)
        return (-(int(m.group(1)) if m else 0), name)

    ordered_members = sorted(members.keys(), key=_rot_key)
    records = []
    seq = 1
    last_ts = None
    for m_name in ordered_members:
        for raw_line in members[m_name].splitlines():
            if not raw_line.strip():
                continue
            line = raw_line.decode("latin1").rstrip("\r\n")
            parts = line.split("\t")
            ts = datetime.strptime(f"{parts[0]} {parts[1]}", "%d/%m/%Y %H:%M:%S")
            level, serial, app, msg = parts[2], parts[3], parts[4], parts[5]
            has_reg = last_ts is not None and ts < last_ts
            last_ts = ts
            ev_code = "UNKNOWN_EVENT"
            attempt_num = None
            for code, rgx, has_attempt in ALLOWLIST_RULES:
                m = rgx.search(msg)
                if m:
                    ev_code = code
                    if has_attempt:
                        attempt_num = int(m.group("attempt"))
                    break
            records.append({
                "election_year": 2026,
                "turn": 1,
                "uf": "SP",
                "municipio_code": 71072,
                "zone_num": 1,
                "section_num": 101,
                "urna_serial": serial,
                "file_seq": seq,
                "raw_timestamp": ts,
                "has_timestamp_regression": has_reg,
                "application": app,
                "log_level": level,
                "event_code": ev_code,
                "biometric_attempt_num": attempt_num,
            })
            seq += 1

    df_events = pl.DataFrame(records, schema=FACT_URNA_EVENT_POLARS_SCHEMA)
    arrow_events = df_events.to_arrow().cast(FACT_URNA_EVENT_ARROW_SCHEMA)
    part_dir = work_dir / "exports" / "fact_urna_event" / "election_year=2026" / "turn=1" / "uf=SP"
    part_dir.mkdir(parents=True, exist_ok=True)
    parquet_path = part_dir / "part-00000.parquet"
    df_events.write_parquet(parquet_path, compression="zstd", compression_level=5)

assert df_events.schema == FACT_URNA_EVENT_POLARS_SCHEMA
assert arrow_events.schema.equals(FACT_URNA_EVENT_ARROW_SCHEMA)
assert df_events["file_seq"].to_list() == list(range(1, 19))

# Verificação binária estrita de Zero-PII e Zero-MAC no Parquet gerado
raw_parquet_bytes = parquet_path.read_bytes()
for sentinel in (PII_SENTINEL_CPF, PII_SENTINEL_TITULO, MAC_SENTINEL, "Operação não catalogada"):
    assert sentinel.encode("utf-8") not in raw_parquet_bytes
    assert sentinel.encode("latin1") not in raw_parquet_bytes

pq_meta = pq.read_metadata(parquet_path)
print(f"Parquet ZSTD gravado com sucesso: {parquet_path}")
print(f"Linhas materializadas: {pq_meta.num_rows} | Compressão: {pq_meta.row_group(0).column(0).compression}")
print(df_events)

Parquet ZSTD gravado com sucesso: /tmp/tse_urna_logs_poc_ub326a71/exports/fact_urna_event/election_year=2026/turn=1/uf=SP/part-00000.parquet
Linhas materializadas: 18 | Compressão: ZSTD
shape: (18, 14)
┌─────────────┬──────┬─────┬─────────────┬───┬─────────────┬───────────┬─────────────┬─────────────┐
│ election_ye ┆ turn ┆ uf  ┆ municipio_c ┆ … ┆ application ┆ log_level ┆ event_code  ┆ biometric_a │
│ ar          ┆ ---  ┆ --- ┆ ode         ┆   ┆ ---         ┆ ---       ┆ ---         ┆ ttempt_num  │
│ ---         ┆ i8   ┆ str ┆ ---         ┆   ┆ str         ┆ str       ┆ str         ┆ ---         │
│ i16         ┆      ┆     ┆ i32         ┆   ┆             ┆           ┆             ┆ i8          │
╞═════════════╪══════╪═════╪═════════════╪═══╪═════════════╪═══════════╪═════════════╪═════════════╡
│ 2026        ┆ 1    ┆ SP  ┆ 71072       ┆ … ┆ scue        ┆ INFO      ┆ LOGD_STARTE ┆ null        │
│             ┆      ┆     ┆             ┆   ┆             ┆           ┆ D           ┆     

In [5]:
# 3. Consultas Analíticas de Verificação via Polars e DuckDB sobre o Parquet Particionado
con = duckdb.connect()
cursor = con.execute(
    f"""
    SELECT
        event_code,
        COUNT(*) AS event_count,
        MIN(file_seq) AS first_seq,
        MAX(file_seq) AS last_seq,
        SUM(CASE WHEN has_timestamp_regression THEN 1 ELSE 0 END) AS regressions,
        MAX(biometric_attempt_num) AS max_bio_attempt
    FROM read_parquet('{parquet_path.as_posix()}')
    GROUP BY event_code
    ORDER BY first_seq
    """
)
cols = [desc[0] for desc in cursor.description]
summary_df = pl.DataFrame(cursor.fetchall(), schema=cols, orient="row")

print("Resumo Analítico por event_code (DuckDB -> Polars):")
print(summary_df)

validation_summary = {
    "status": "SUCCESS",
    "execution_mode": EXECUTION_MODE,
    "total_events": int(df_events.height),
    "regression_file_seqs": df_events.filter(pl.col("has_timestamp_regression"))["file_seq"].to_list(),
    "distinct_event_codes": int(summary_df.height),
    "zero_pii_verified": True,
    "parquet_compression": pq_meta.row_group(0).column(0).compression,
}
Path("poc_validation_summary.json").write_text(json.dumps(validation_summary, indent=2), encoding="utf-8")
print("Resumo de validação salvo em poc_validation_summary.json:")
print(json.dumps(validation_summary, indent=2))

Resumo Analítico por event_code (DuckDB -> Polars):
shape: (14, 6)
┌──────────────────────────┬─────────────┬───────────┬──────────┬─────────────┬─────────────────┐
│ event_code               ┆ event_count ┆ first_seq ┆ last_seq ┆ regressions ┆ max_bio_attempt │
│ ---                      ┆ ---         ┆ ---       ┆ ---      ┆ ---         ┆ ---             │
│ str                      ┆ i64         ┆ i64       ┆ i64      ┆ i64         ┆ i64             │
╞══════════════════════════╪═════════════╪═══════════╪══════════╪═════════════╪═════════════════╡
│ LOGD_STARTED             ┆ 1           ┆ 1         ┆ 1        ┆ 0           ┆ null            │
│ SECTION_OPENED           ┆ 2           ┆ 2         ┆ 3        ┆ 0           ┆ null            │
│ VOTER_ENABLED            ┆ 2           ┆ 4         ┆ 13       ┆ 0           ┆ null            │
│ BIOMETRIC_READ_ATTEMPT   ┆ 2           ┆ 5         ┆ 7        ┆ 0           ┆ 2               │
│ BIOMETRIC_NOT_RECOGNIZED ┆ 1           ┆ 6       

## 4. Inspeção de Log Real das Eleições 2026 (`HEAD` e `TAIL` de `logd.dat` e `fact_urna_event`)

Abaixo extraímos via **HTTP Range** diretamente do pacote oficial de Arquivos Transmitidos para Totalização do TSE (`bu_imgbu_logjez_rdv_vscmr_2026_1t_AC.zip`) um único arquivo `.jez` real (`o03220ac0101500020088-log.jez`, ~68 KB, Município `01015`, Zona `0002`, Seção `0088` — Acre, 1º Turno 2026) sem baixar o pacote inteiro de ~393 MB.

Exibimos:
1. **`HEAD (15 primeiras linhas)`** e **`TAIL (15 últimas linhas)`** do `logd.dat` real (com `mac_code` e identificadores numéricos longos mascarados por segurança).
2. **Amostra de sessão real de votação (`file_seq` 706 a 722)**.
3. **`HEAD (15)`**, **`TAIL (15)`** e distribuição por `event_code` da tabela `fact_urna_event` materializada a partir do `.jez` real.

In [6]:
import struct
import zlib

import httpx

TSE_2026_AC_URL = "https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes2026/arqurnatot/bu_imgbu_logjez_rdv_vscmr_2026_1t_AC.zip"

def _fetch_single_real_jez(url: str) -> tuple[str, bytes]:
    with httpx.Client(timeout=30.0, follow_redirects=True) as client:
        size = int(client.head(url).headers["content-length"])
        tail_len = min(size, 2 * 1024 * 1024)
        tail = client.get(url, headers={"Range": f"bytes={size - tail_len}-{size - 1}"}).content
        eocd_pos = tail.rfind(b"PK\x05\x06")
        cd_size, cd_offset = struct.unpack("<II", tail[eocd_pos + 12 : eocd_pos + 20])
        cd = client.get(url, headers={"Range": f"bytes={cd_offset}-{cd_offset + cd_size - 1}"}).content
        pos = 0
        while pos + 46 <= len(cd):
            if cd[pos : pos + 4] != b"PK\x01\x02":
                break
            comp_method = struct.unpack("<H", cd[pos + 10 : pos + 12])[0]
            comp_size, _, fn_len, extra_len, comment_len = struct.unpack("<IIHHH", cd[pos + 20 : pos + 34])
            lhoff = struct.unpack("<I", cd[pos + 42 : pos + 46])[0]
            fname = cd[pos + 46 : pos + 46 + fn_len].decode("latin1")
            pos += 46 + fn_len + extra_len + comment_len
            if fname.lower().endswith(".jez"):
                raw_entry = client.get(url, headers={"Range": f"bytes={lhoff}-{lhoff + 30 + len(fname) + 256 + comp_size}"}).content
                fn_l, ex_l = struct.unpack("<HH", raw_entry[26:30])
                comp_data = raw_entry[30 + fn_l + ex_l : 30 + fn_l + ex_l + comp_size]
                jez_payload = zlib.decompress(comp_data, -zlib.MAX_WBITS) if comp_method == 8 else comp_data
                return fname, jez_payload
    raise RuntimeError("Nenhum .jez encontrado no pacote remoto.")

def _sanitize_line_preview(line_str: str) -> str:
    parts = line_str.rstrip("\r\n").split("\t")
    if len(parts) >= 2:
        parts[-1] = "<MAC_REDACTED>"
    if len(parts) >= 5:
        parts[-2] = re.sub(r"\b\d{5,}\b", "<ID_REDACTED>", parts[-2])
    return "\t".join(parts)

real_jez_name, real_jez_bytes = _fetch_single_real_jez(TSE_2026_AC_URL)
print(f"Arquivo .jez real obtido via HTTP Range: {real_jez_name} ({len(real_jez_bytes):,} bytes)")

with zipfile.ZipFile(io.BytesIO(real_jez_bytes), mode="r") as zf:
    real_raw_lines = [ln.decode("latin1") for ln in zf.read("logd.dat").splitlines() if ln.strip()]

print(f"Total de linhas fisicamente gravadas em logd.dat: {len(real_raw_lines):,}\n")
print("=== HEAD (15 primeiras linhas do logd.dat real — MAC/IDs sanitizados) ===")
for idx, ln in enumerate(real_raw_lines[:15], start=1):
    print(f"{idx:4d} | {_sanitize_line_preview(ln)}")

print("\n=== AMOSTRA DE SESSÃO DE VOTAÇÃO REAL (file_seq 706..722 — VOTA) ===")
for idx in range(706, 723):
    print(f"{idx:4d} | {_sanitize_line_preview(real_raw_lines[idx - 1])}")

print("\n=== TAIL (15 últimas linhas do logd.dat real — MAC/IDs sanitizados) ===")
total_ln = len(real_raw_lines)
for idx, ln in enumerate(real_raw_lines[-15:], start=total_ln - 14):
    print(f"{idx:4d} | {_sanitize_line_preview(ln)}")

# Materialização em fact_urna_event e exibição de head/tail + distribuição de event_code
if EXECUTION_MODE == "local_package":
    from tse_analysis.logs import parse_section_keys_from_filename
    real_ctx = parse_section_keys_from_filename(real_jez_name, election_year=2026, turn=1, uf="AC")
    real_ext = extract_jez_events(real_jez_bytes, section_context=real_ctx, availability_status="AVAILABLE")
    df_real_events = events_to_polars([real_ext])
else:
    real_records = []
    last_ts = None
    for seq_idx, raw_ln in enumerate(real_raw_lines, start=1):
        parts = raw_ln.rstrip("\r\n").split("\t")
        ts = datetime.strptime(parts[0].strip(), "%d/%m/%Y %H:%M:%S")
        level, serial, app, msg = parts[1].strip(), parts[2].strip(), parts[3].strip(), parts[4].strip()
        has_reg = last_ts is not None and ts < last_ts
        last_ts = ts
        ev_code = "UNKNOWN_EVENT"
        attempt_num = None
        for code, rgx, has_attempt in ALLOWLIST_RULES:
            m = rgx.search(msg)
            if m:
                ev_code = code
                if has_attempt:
                    attempt_num = int(m.group("attempt"))
                break
        real_records.append({
            "election_year": 2026,
            "turn": 1,
            "uf": "AC",
            "municipio_code": 1015,
            "zone_num": 2,
            "section_num": 88,
            "urna_serial": serial,
            "file_seq": seq_idx,
            "raw_timestamp": ts,
            "has_timestamp_regression": has_reg,
            "application": app,
            "log_level": level,
            "event_code": ev_code,
            "biometric_attempt_num": attempt_num,
        })
    df_real_events = pl.DataFrame(real_records, schema=FACT_URNA_EVENT_POLARS_SCHEMA)

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(14)
print("\n=== HEAD (15 primeiros registros) de fact_urna_event (Log Real 2026 AC) ===")
print(df_real_events.head(15))

print("\n=== TAIL (15 últimos registros) de fact_urna_event (Log Real 2026 AC) ===")
print(df_real_events.tail(15))

print("\n=== Distribuição de event_code no Log Real 2026 AC (o03220ac0101500020088-log.jez) ===")
print(
    df_real_events.group_by("event_code")
    .agg([
        pl.len().alias("count"),
        pl.col("file_seq").min().alias("first_seq"),
        pl.col("file_seq").max().alias("last_seq"),
        pl.col("biometric_attempt_num").max().alias("max_attempt"),
    ])
    .sort("count", descending=True)
)

Arquivo .jez real obtido via HTTP Range: o03220ac0101500020088-log.jez (68,091 bytes)
Total de linhas fisicamente gravadas em logd.dat: 3,714

=== HEAD (15 primeiras linhas do logd.dat real — MAC/IDs sanitizados) ===
   1 | 23/09/2026 10:39:02	INFO	02263093	LOGD	Início das operações do logd	<MAC_REDACTED>
   2 | 23/09/2026 10:39:02	INFO	02263093	LOGD	Urna ligada em 23/09/2026 às 10:38:08	<MAC_REDACTED>
   3 | 23/09/2026 10:39:02	INFO	02263093	SCUE	Iniciando aplicação - 1º turno	<MAC_REDACTED>
   4 | 23/09/2026 10:39:02	INFO	02263093	SCUE	Versão da aplicação: 10.23.0.0 - Praia da Barra do Cahy	<MAC_REDACTED>
   5 | 23/09/2026 10:39:04	INFO	02263093	SCUE	Urna operando com rede elétrica	<MAC_REDACTED>
   6 | 23/09/2026 10:39:04	INFO	02263093	SCUE	Bateria interna com carga plena	<MAC_REDACTED>
   7 | 23/09/2026 10:39:10	INFO	02263093	SCUE	Tamanho da mídia interna: 3.7 GB	<MAC_REDACTED>
   8 | 23/09/2026 10:39:13	INFO	02263093	SCUE	Tamanho da memória: 3.7 GB	<MAC_REDACTED>
   9 | 23/09/2026

shape: (14, 5)
┌──────────────────────────┬───────┬───────────┬──────────┬─────────────┐
│ event_code               ┆ count ┆ first_seq ┆ last_seq ┆ max_attempt │
│ ---                      ┆ ---   ┆ ---       ┆ ---      ┆ ---         │
│ str                      ┆ u32   ┆ i64       ┆ i64      ┆ i8          │
╞══════════════════════════╪═══════╪═══════════╪══════════╪═════════════╡
│ UNKNOWN_EVENT            ┆ 1549  ┆ 2         ┆ 3714     ┆ null        │
│ CARGO_VOTE_RECORDED      ┆ 846   ┆ 662       ┆ 3654     ┆ null        │
│ BIOMETRIC_READ_ATTEMPT   ┆ 538   ┆ 658       ┆ 3646     ┆ 4           │
│ INVALID_KEY_PRESSED      ┆ 325   ┆ 666       ┆ 3638     ┆ null        │
│ VOTER_ENABLED            ┆ 141   ┆ 661       ┆ 3648     ┆ null        │
│ VOTER_VOTE_COMPUTED      ┆ 141   ┆ 669       ┆ 3655     ┆ null        │
│ BIOMETRIC_RECOGNIZED     ┆ 122   ┆ 660       ┆ 3647     ┆ null        │
│ MANUAL_ENABLE_PATH       ┆ 21    ┆ 688       ┆ 3282     ┆ null        │
│ INACTIVITY_TIMEOUT_45